# 歌词采集

In [32]:
import requests
import re
import json
import os
import time
import pandas as pd
from datetime import datetime
from collections import defaultdict


from collections import Counter

In [2]:
import sys
sys.path.append('..')

# 通用方法

## 接口
专辑: https://music.163.com/api/album/{album_id}
歌词：https://music.163.com/api/song/lyric?id={song_id}&lv=1&kv=1&tv=-1

In [43]:
headers = {
    # "Accept": "*/*",
    # "Accept-Encoding": "gzip, deflate, br",
    # "Accept-Language": "zh-CN,zh;q=0.9,en;q=0.8,en-GB;q=0.7,en-US;q=0.6",
    # "Connection": "keep-alive",
    # "Content-Length": "618",
    # "Content-Type": "text/html;charset=UTF-8",
    "Cookie": "_iuqxldmzr_=32; _ntes_nnid=417b035df23de02a82843d9de05e8d59,1769240680818; _ntes_nuid=417b035df23de02a82843d9de05e8d59; NMTID=00ORE6BRHoa0Ify9Uc3hX1wbRioP1sAAAGb7vZZ0g; Hm_lvt_d94b7d26fa25db7f6413fb58d7a438c7=1769240681; HMACCOUNT=0C6E31D9EADCB199; WEVNSM=1.0.0; WNMCID=zludjp.1769240681856.01.0; WM_TID=g8%2FPMLrhV8xEUVVBUVbXiqjt62wjuVTz; sDeviceId=YD-TvaxP%2B9vZedFR1VRFVKW2ry87ixpBqKq; ntes_utid=tid._.NeSaF4EyJGNFUgBAFFOXjqnpry0tUsta._.0; __snaker__id=NbSHzHH3f7FFoBev; ntes_kaola_ad=1; _ntes_origin_from=; NTES_P_UTID=HXQrvJaNkn2XUZ3kdY6QQdwPMWItbY28|1769584953; NTES_SESS=bSvX3oZr5dfnfmhI88Js6Becf0JSu2bl5XfY6aiJKRwoNK67NXivTdl8ZVCT5qqJbJZW3QQNsM5Wp29Ca0t5Uywyc0QG0ylZixUdmz_UxbXx_f4ArYqq.D._E8bV_BW9qeo9r3fcN4mIb6P503JitfjUlclSN_WhDwH6tYZGOgy0OHGlpguiNwGoReSbcU97p4bxLeHEvYQQd; S_INFO=1769584953|0|3&80##|wangkefeicn; P_INFO=wangkefeicn@163.com|1769584953|0|mail163|00&18|sic&1717721610&mail163#sic&510100#10#0#0|&0||wangkefeicn@163.com; __csrf=52dbec9dc637200b228ee473e917b727; MUSIC_U=003FEF704A65165E462988FF7406BD4A76026EC0ED427FB04AA9E9A6641FBC3697B9EF31F134AF6AB52C7089DC2BB83E9829F0823CCDC3B3AE3CD94C2BB8CF7430F43E03EC590425828339D54329803A483D567A3D90279B832EEC76597F64B95D5900E3425039E5D6333B2EF2FBCD73AF77C8F928A4795E4605A5EF16B25CCD9A51D268B8549DD344CA35FD6AC64E031CD60214C3B92A0EB2FFA82D2B2D02E0276DB6301C431BF5B3B4551B17A72401604A537E08FE3E2CDD07FD397A8E97E230083ABC70E4ADEB5C27CFD53F2412E074A0DC7D68B8F6C3DADA025A0B4487D855E765A692E2D303F93212386BA517886AE585188F1710871BCED6752F4E9C885019B546439811E95CB1FA57A9F27606C442EA1FE4A901586271291D3EEB14F409069FD9DBE2E0DE40E74510CC88B1A26ABAE7FA990618958A89F6F4088A913326AB72CA1B0091CB6F652DC86F9BD4EE82E30A7A71B1FBB98C36F2E50543D3084EBA3C37C772BEF9273ABD1917466C83E7BB101285477AC2C1C988A99C804AA2E58F5CC1B5F4B4A354D91DBC5C84EE15ABF55D4FC29E89E3BCE0409D6E8B8EEFAC8D580F474AAD8B70588292046DD30978; gdxidpyhxdE=Asu0EwLkjPIZrUZ0KrZa3QXSGnarnbMaOQX4D6I4k7BKnn%2Bbzxs%2BKcfr%2F8zlirkSST1idpeYQcJLaWQ9iyH%5CVd2YcWCR1hzJ1n5ZjpgW%2B63JJCRehpKzl1YeySo%2FDmXiABm3zwWII90iE%5CD0IOgVxuheyoOH7JQH%2BE0y3w0cwS3%2Fz%5Ca9%3A1770015350929; Hm_lpvt_d94b7d26fa25db7f6413fb58d7a438c7=1770101385; WM_NI=uSgU3ifd8wRH38%2BrOeVbDW5LjxXyNGP6WKixG0pq28r2CzxwC55Hqq5AHHgfz%2BfSQ39BW4jFZtsmAOcbL%2Bry2Z6aL%2B%2FSFSKh%2F9e97Jp%2Fox4UWXVZ59A4fRN0uV1nEKW1eTg%3D; WM_NIKE=9ca17ae2e6ffcda170e2e6eed5ef5f94b7b78fb750908e8aa2d85f869f9bb0d67986b8b68bc974a1bdae90c72af0fea7c3b92ab78da9b6b263a6bcf898d144f1e8e593ce6f96be8cb8b848aaa8989acd4d94ea99acec7485f0c092b134b2ee8d9bfb7d81ebfba9f47bf79f88a9e66ebaf5aab4d439fb9e99d5e94e9aacbd8ec445a1be84ccf173f286fa82cf21ed8d8ab4ca64ede7abdab642b4f1a199b75f95889abbb83383ecaab3cd3cb78aff8fc84195efabb5e637e2a3; Hm_lvt_1483fb4774c02a30ffa6f0e2945e9b70=1770195300; JSESSIONID-WYYY=%5CnwhV%5Cek%2FDMJXvDNxdsVYTj%2F%5Cv0ECTpQvOZ6dywQl4H%2FyIpYHXDmvbkVcG5n0OxiZ2n%5C%5C%5CN5S2XNm1Qrr0Ip2jK6ZwHJpebHvWb36xRUEk3TsnaR9FVfvBQp7ek%2FJMytHGsIfQEkJU0OO09%2B5nK9qh4stH5%2B9oSd%5C%2BZflusfWOV8bmDg%3A1770199039356; Hm_lpvt_1483fb4774c02a30ffa6f0e2945e9b70=1770197730",
    # "Referer": "https://music.163.com/search/",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/87.0.4280.88 Safari/537.36 Edg/87.0.664.66",}

In [33]:
def format_timestamp(ts_ms, date_format='%Y-%m-%d'):
    """
    将毫秒级时间戳转换为指定格式的年月日字符串
    :param ts_ms: 13位时间戳 (int 或 float)
    :param date_format: 输出格式，默认为 'YYYY-MM-DD'
    :return: 格式化后的日期字符串
    """
    if not ts_ms or ts_ms <= 0:
        return "Unknown"
    
    try:
        # 将毫秒转为秒并生成日期对象
        dt = datetime.fromtimestamp(ts_ms / 1000)
        return dt.strftime(date_format)
    except Exception:
        return "Invalid Date"

# 专辑采集
album_api = "https://music.163.com/api/album/{album_id}"

In [46]:
def get_album_data(album_id, headers):
    api_url = f"https://music.163.com/api/album/{album_id}"

    try:
        response = requests.get(api_url, headers=headers, timeout=10)
        response.raise_for_status()
        data = response.json()
        return data
    except Exception as e:
        print(f"ID {album_id} 获取失败: {e}")
        return ""

In [48]:
album_id_test = "120613822"
album_data = get_album_data(album_id_test, headers)
format_timestamp(album_data['album']['publishTime'])

'2020-12-19'

# 曲目采集-按歌手搜索结果

## API 
因参数加密问题，通过脚本采集数据的话，每次都要去浏览器复制param和enSecKey值，非常麻烦。

In [ ]:
songs_api = "https://music.163.com/weapi/cloudsearch/get/web?csrf_token=52dbec9dc637200b228ee473e917b727"

payload = {
    'params': 'S1yVLSGVT4ARnDG91/z+Xy4wPRAw4JeSDOk9yzqGc52AaAu4K4Pj4G+c1eK1cs5b5x5F+eSoQFPwe0qMmFyhMiiROfIc5gy82s3thLAKzvqXXyBBkPwEABEBE18j9OxseLujpt8b9/7gunGHKyOk2Uycqvxem/+uioRR1s/C2zAn/LJ4vPjcUszoWbcVe+A+L2ySQd863pUPzgckgn9XhoL/vlffrBciB6X144DYHcrPrnSalb0C6VYaa7AVxMTblWATVc1YHVwlRNQwZlbxtwccATe93s4Lrb051Rk2g6e95v7tB7Z9EqLezVs0MWXY',
    "encSecKey": '2c7698452d0c2aeb455f5a491e36c31935b076b2b2971296e6ed122d60c820d5bb8608738a9fcb672a84b143636f8c16926bfce1267031d0e96da0354f957a74ab8df40c20a15acadf6e470b7a23b1d504b64ce588d9ba4d251f3c722283f15c25bfbe9f102f50e7a4d8bcab4bde3044fe37b702166600ac68e985a8fddf07de',
    'offset': 30,
}

In [ ]:
# 未启用
# response = requests.post(songs_api, data=payload, headers=headers)
# raw_text = response.content.decode('utf-8', errors='ignore')

# # 2. 将字符串解析为 Python 字典
# raw_data = json.loads(raw_text)
# raw_songs = raw_data['result']['songs']
# len(raw_songs)


## 手工采集
从浏览器->开发者工具->网络->Fetch/XHR->web?csrf_token=52dbec9dc637200b228ee473e917b727->响应->复制->保存到json文件的列表中

In [50]:
def extract_music_data_from_raw(data):
    songs = []
    for song_set in data:
        songs.extend(song_set.get('result', {}).get('songs', []))

    song_list = []
    for s in songs:
        # Extract aliases/subnames from the 'alia' list
        aliases = s.get('alia', [])
        song_subname = " / ".join(aliases) if aliases else ""

        # Artist and Album extraction
        artists = s.get('ar', [])
        artist_names = [a.get('name') for a in artists]

        album = s.get('al', {})

        # Duration conversion
        dt = s.get('dt', 0)
        duration_formatted = f"{int(dt//60000):02d}:{int((dt//1000)%60):02d}"

        # Building the record
        song_info = {
            "song_name": s.get('name'),
            "song_subname": song_subname,  # Added alia mapping here
            "song_id": str(s.get('id')),
            "artist_name": "/".join(artist_names),
            "artist_ids": [str(a.get('id')) for a in artists],
            "album_name": album.get('name') or "Single",
            "album_id": str(album.get('id')),
            "duration": duration_formatted,
            "pic_url": album.get('picUrl')
        }
        song_list.append(song_info)

    return song_list

In [51]:
# 原始数据文件
file_path = "html_data_raw/lyn/raw_data_songs.json"
raw_data_songs = json.load(open(file_path, 'r', encoding='utf-8'))
# 解析
song_data_0 = extract_music_data_from_raw(raw_data_songs)
len(song_data_0)

210

In [52]:
song_data_0[0]

{'song_name': '当遇见你',
 'song_subname': '电视剧《冰糖炖雪梨》片尾曲',
 'song_id': '1426285166',
 'artist_name': '刘宇宁',
 'artist_ids': ['1094010'],
 'album_name': '当遇见你',
 'album_id': '86025678',
 'duration': '03:11',
 'pic_url': 'http://p1.music.126.net/G_2C4j_g-vC_3M2YJ-F5pg==/109951164744571789.jpg'}

In [53]:
# 数据筛选
# 1. artist_name中包含"刘宇宁"
# 2. song_subname中包含书名号，将书名号中的内容保存为新字段: tv_name
def filter_songs(singger, song_list):
    """
    筛选符合条件的歌曲：
    1. 歌手包含 "刘宇宁"
    2. 子标题包含书名号，并提取书名号内容为 tv_name
    """
    filtered_list = []
    # 预编译正则，匹配《 和 》之间最少的内容
    tv_pattern = re.compile(r'《(.*?)》')

    for song in song_list:
        # 条件 1: 校验 artist_name (确保该字段已在之前的解析中生成)
        if singger not in song.get("artist_name", ""):
            continue
            
        # 条件 2: 校验 song_subname 并在满足时提取 tv_name
        subname = song.get("song_subname", "")
        match = tv_pattern.search(subname)
        
        if match:
            # 满足条件，创建新字段并保存
            song["tv_name"] = match.group(1)
            filtered_list.append(song)
            
    return filtered_list


In [54]:
singer = "刘宇宁"
song_data_1 = filter_songs(singer, song_data_0)
len(song_data_1)

67

In [62]:
song_data_1[33]


{'song_name': '年长',
 'song_subname': '《蜀锦人家》影视剧插曲',
 'song_id': '2652860446',
 'artist_name': '刘宇宁',
 'artist_ids': ['1094010'],
 'album_name': 'Single',
 'album_id': '0',
 'duration': '03:01',
 'pic_url': 'http://p2.music.126.net/VNxaQ3BnsVYQJnlIEJ8iFA==/109951170220123707.jpg',
 'tv_name': '蜀锦人家'}

In [63]:
for i in song_data_1:
    print(i['album_id'])

86025678
78192180
72071715
271800185
179878642
159617297
72977504
170736863
120613822
92626041
179878642
180029497
124017766
135435127
126195934
129362310
269091359
95443275
289152637
83604634
153102651
276569050
189025323
131501916
272874488
152360750
274124607
176513988
274124607
98475930
245650871
124758577
241969288
0
122577492
169269527
157765273
244934802
248804632
121899191
72977172
163594572
139592400
88615616
74149619
247835321
268486346
126355262
96032308
82590038
122710363
187452798
94508741
136946487
164534947
161917036
180029497
360718069
358291522
355198393
277024417
277024417
288261528
124988859
189981796
283198943
349046699


In [64]:
def add_release_date_to_songs(song_data, headers):
    res = []
    album_dict = defaultdict(str)
    album_dict['0'] = ""
    i_num = 0
    for i in song_data:
        album_id = i['album_id']
        print(i_num, album_id, i['album_name'])
        i_num += 1
        if album_id in album_dict:
            i['release_date'] = album_dict[album_id]
        else:
            album_data = get_album_data(album_id, headers)
            release_date = format_timestamp(album_data['album']['publishTime'])
            i['release_date'] = release_date
            album_dict[album_id] = release_date
        res.append(i)
    return res

In [65]:
song_data_final = add_release_date_to_songs(song_data_1, headers)

0 86025678 当遇见你
1 78192180 我们的师父
2 72071715 让酒
3 271800185 折腰 影视原声专辑
4 179878642 《一念关山》影视剧原声大碟
5 159617297 《星落凝成糖》电视原声音乐专辑
6 72977504 有多少爱可以重来
7 170736863 《安乐传》影视原声带
8 120613822 终极笔记 影视原声带
9 92626041 我的漂亮朋友 影视原声带
10 179878642 《一念关山》影视剧原声大碟
11 180029497 龙吟
12 124017766 司藤 影视原声带
13 135435127 千里江山
14 126195934 八零九零 影视原声带
15 129362310 千古玦尘 电视剧影视原声带
16 269091359 那些喊给天空的
17 95443275 太古神王 影视原声带
18 289152637 《水龙吟》影视原声大碟
19 83604634 十
20 153102651 《追光者》电视剧原声专辑
21 276569050 以法之名 电视剧原声带
22 189025323 《烈焰》影视原声带
23 131501916 破晓
24 272874488 我只愿
25 152360750 阳光、海浪、我和你
26 274124607 说英雄谁是英雄 网剧原声带
27 176513988 《他从火光中走来》影视剧原声带
28 274124607 说英雄谁是英雄 网剧原声带
29 98475930 最初的相遇，最后的别离 影视原声带
30 245650871 《赘婿》动画原声带
31 124758577 你是我的城池营垒 电视剧影视原声带
32 241969288 《错位》影视原声带
33 0 Single
34 122577492 我的时代，你的时代 电视剧影视原声带
35 169269527 我的人间烟火 电视剧原声带
36 157765273 向风而行 影视剧原声带
37 244934802 《孤舟》电视原声带
38 248804632 《暗夜与黎明》电视原声带
39 121899191 假日暖洋洋 电视原声带
40 72977172 探险家
41 163594572 电视剧 春闺梦里人 影视原声大碟
42 139592400 好运歌
43 88615616 全世界最好

In [66]:
song_data_final

[{'song_name': '当遇见你',
  'song_subname': '电视剧《冰糖炖雪梨》片尾曲',
  'song_id': '1426285166',
  'artist_name': '刘宇宁',
  'artist_ids': ['1094010'],
  'album_name': '当遇见你',
  'album_id': '86025678',
  'duration': '03:11',
  'pic_url': 'http://p1.music.126.net/G_2C4j_g-vC_3M2YJ-F5pg==/109951164744571789.jpg',
  'tv_name': '冰糖炖雪梨',
  'release_date': '2020-02-27'},
 {'song_name': '我们的师父',
  'song_subname': '综艺《我们的师父》同名主题曲',
  'song_id': '1354206852',
  'artist_name': '大张伟/于晓光/刘宇宁/WINWIN(董思成)',
  'artist_ids': ['2524', '12322210', '1094010', '31757669'],
  'album_name': '我们的师父',
  'album_id': '78192180',
  'duration': '02:40',
  'pic_url': 'http://p1.music.126.net/Y1wIUWDhmwX6GhMi-5UTaQ==/109951163950420722.jpg',
  'tv_name': '我们的师父',
  'release_date': '2019-03-25'},
 {'song_name': '让酒',
  'song_subname': '电视剧《沙海》插曲',
  'song_id': '1297742298',
  'artist_name': '刘宇宁',
  'artist_ids': ['1094010'],
  'album_name': '让酒',
  'album_id': '72071715',
  'duration': '04:26',
  'pic_url': 'http://p1.music.126.

# 歌词采集

In [68]:
# --- 核心清洗函数 ---
def clean_and_format_lyrics(raw_text):
    if not raw_text or "未能获取" in raw_text:
        return ""

    # 1. 预处理：处理特殊空格 U+00A0
    text = raw_text.replace('\u00a0', ' ')

    # 2. 定义黑名单关键词
    exclude_keywords = [
        '作词', '作曲', '编曲', '：', ':', '演奏', '吉他', '贝斯',
        '鼓', '编写', '演唱', '合唱', '制作', '录音', '混音', 'ISRC',
        '编码', '版权', '提供', '发行', 'OP', 'SP'
    ]

    # 3. 专门针对 ISRC 这种特征码的正则表达式
    # 匹配规律：大写字母开头，中间有多个连字符和数字，例如 TW-K23-08-016-11
    isrc_pattern = r'[A-Z]{2}-[A-Z0-9]{3}-\d{2}-\d{5}'

    pure_lyrics_list = []
    lines = text.split('\n')

    for line in lines:
        # 提取时间戳后面的内容
        match = re.search(r'\[.*\]\s*(.*)', line)
        if match:
            content = match.group(1).strip()

            # --- 过滤逻辑开始 ---
            # A. 检查是否为空
            if not content:
                continue

            # B. 检查是否包含黑名单关键词
            if any(k in content.upper() for k in exclude_keywords): # 转大写匹配，防止漏掉 isrc
                continue

            # C. 检查是否匹配 ISRC 正则特征
            if re.search(isrc_pattern, content):
                continue

            # --- 过滤逻辑结束 ---

            # 内部空格换逗号，去除多余空白
            clean_content = re.sub(r'\s+', ' ', content).replace(" ", "，")
            pure_lyrics_list.append(clean_content)

    # 用句号连接
    if not pure_lyrics_list: return ""
    return "。".join(pure_lyrics_list) + "。"

# --- 制作信息提取 ---
def get_credit_info(raw_text):
    """从原始文本中提取 作词/作曲/编曲"""
    info = {"作词": "", "作曲": "", "编曲": ""}
    # 兼容带时间戳和不带时间戳的情况
    for key in info.keys():
        pattern = rf"{key}\s*[:：]\s*([^\]\n]+)"
        match = re.search(pattern, raw_text)
        if match:
            # 清理掉可能残余的括号或空格
            info[key] = match.group(1).strip()
    return info

# --- API 请求函数 ---
def get_lyrics_by_api(song_id):
    api_url = f"https://music.163.com/api/song/lyric?id={song_id}&lv=1&kv=1&tv=-1"
    headers = {"User-Agent": "Mozilla/5.0"}

    try:
        response = requests.get(api_url, headers=headers, timeout=10)
        response.raise_for_status()
        data = response.json()
        return data.get('lrc', {}).get('lyric', "")
    except Exception as e:
        print(f"ID {song_id} 获取失败: {e}")
        return ""

# --- 主逻辑封装 ---
def process_single_song(song_id, song_name):
    """处理单首歌曲：下载 -> 解析 -> 组装字典"""
    raw_lyric = get_lyrics_by_api(song_id)

    # 提取制作人信息
    credits = get_credit_info(raw_lyric)
    # 提取并清洗歌词
    formatted_lyric = clean_and_format_lyrics(raw_lyric)

    has_lyric = 1
    if not raw_lyric:
        has_lyric = 0
        formatted_lyric = ""
    # 组装结果
    return {
        "歌名": song_name,
        "song_id": song_id,
        "has_lyric": has_lyric,
        "作词": credits["作词"],
        "作曲": credits["作曲"],
        "编曲": credits["编曲"],
        "歌词": formatted_lyric
    }

# --- 文件保存（增量更新） ---
def save_to_json_list(file_path, song_data):
    """以列表形式保存所有歌曲，避免字典 key 覆盖的问题"""
    data_list = []
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            try:
                data_list = json.load(f)
                if not isinstance(data_list, list): data_list = []
            except:
                data_list = []

    data_list.append(song_data)

    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(data_list, f, ensure_ascii=False, indent=4)

In [71]:
# 测试
song_id_test = '2753868915'
song_name_test = '不沐春风不遇你'
process_single_song(song_id_test, song_name_test)

{'歌名': '不沐春风不遇你',
 'song_id': '2753868915',
 'has_lyric': 0,
 '作词': '',
 '作曲': '',
 '编曲': '',
 '歌词': ''}

In [77]:
file_path = 'data_output/liuyuning/lyric_data.json'
# 遍历df_df_unique每一行
# 每次运行前删除file_path文件
# 运行结束后，将json内容手动复制到主文件
for song in song_data_final:
    song_id = song['song_id']
    song_name = song['song_name']
    # 判断是否已采集
    if os.path.exists(file_path):
        df = pd.read_json(file_path)
        songs_had = df['song_id'].astype(str).tolist()
        if song_id not in songs_had:
            print(song_name)
            single_res = process_single_song(song_id, song_name)
            save_to_json_list(file_path, single_res)
            time.sleep(2)
    else:
        print(song_name)
        single_res = process_single_song(song_id, song_name)
        save_to_json_list(file_path, single_res)
        time.sleep(2)

In [81]:
df = pd.read_json(file_path)
df[df['has_lyric'] == 0]

,歌名,song_id,has_lyric,作词,作曲,编曲,歌词
30,隐侠 (《赘婿》动画片头曲),2040060429,0,,,,
57,荣光,3346056557,0,,,,
62,毒酒,2753868915,0,,,,
66,万剑不改,2759832433,0,,,,
